In [1]:
import re
import os
import json
from nltk.stem import PorterStemmer
import nltk

nltk.download('punkt')
#stemming kelie we are using porter stemmer
stemmer = PorterStemmer()

print("All imports successful!")

[nltk_data] Downloading package punkt to C:\Users\Ibad Ur
[nltk_data]     Rehman\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


All imports successful!


In [5]:
#loading stopwords from the given file

#why we are using set? 
    # bcs lookup O(1) hoga through set data structure

def load_stopwords(filepath):
    with open(r'C:\Users\Ibad Ur Rehman\Desktop\IR_Assignment1\stopwords.txt', 'r', encoding='utf-8') as f:stopwords = set(line.strip().lower() for line in f if line.strip())
    return stopwords

#simply jo stopwords load kiey hain unhe print krdou...
stopwords = load_stopwords("stopwords.txt")
print(f"Loaded {len(stopwords)} stopwords:")
print(stopwords)

Loaded 26 stopwords:
{'up', 'are', 'her', 'his', 'be', 'can', 'have', 'on', 'is', 'as', 'a', 'am', 'at', 'and', 'do', 'to', 'has', 'for', 'all', 'we', 'in', 'no', 'of', 'once', 'had', 'the'}


In [41]:
#preprocessing here

#steps?? 
    # 1.tokenize 2.lowercase -> 3.remove stopwords -> 4.stemming
def tokenize(text):
    #regex ka concept use krte hoe we eliminate all punctuation and spaces and break sentences into words
    return re.findall(r'[a-zA-Z]+', text)

def preprocess(text, stopwords):

    #tokenize
    tokens = tokenize(text)
    #conversion into lowercase
    tokens = [t.lower() for t in tokens]
    #agar stopword nahi hai tou keep that in tokens
    tokens = [t for t in tokens if t not in stopwords]
    #now apply stemming
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

#now this function is for upcoming user query
def preprocess_query_term(term):

    #we will treat the query term as same as we did to our dictionary term. 
    #why strip()?
        #removes extra spaces
    term = term.lower().strip()
    if term in stopwords:
        return None
    return stemmer.stem(term)

print("Yayyyyy,Preprocessing functions defined!!!!")

def preprocess_with_positions(text, stopwords):

    #Tokenizes and stems but keeps stopwords in place as None so that position numbers reflect original word positions. Used for building the positional index only.
    tokens = tokenize(text)
    result = []
    for position, token in enumerate(tokens):
        lower = token.lower()
        if lower in stopwords:
            result.append((position, None))  #placeholder, keeps position count accurate
        else:
            result.append((position, stemmer.stem(lower)))
    return result

print("preprocess_with_positions defined!")

Yayyyyy,Preprocessing functions defined!!!!
preprocess_with_positions defined!


In [9]:
#let's check k sentence preprocess honay k baad kesa lagay ga.
test_sentence = "Donald Trump is working for becoming President of the United States"
result = preprocess(test_sentence, stopwords)
print("Original:", test_sentence)
print("Processed:", result)

Original: Donald Trump is working for becoming President of the United States
Processed: ['donald', 'trump', 'work', 'becom', 'presid', 'unit', 'state']


In [23]:
#ub yahan se inverted index bananay ka kaam shuru hoga
#we read every speech file and store its text with its document ID.

def load_documents(folder_path):
    """
    Reads all speech_X.txt files from the folder.
    Returns a dict: {doc_id (int): raw_text (str)}
    """
    documents = {}
    for filename in os.listdir(folder_path):
        if filename.endswith('.txt'):
            # Extract the number from filename e.g. speech_3.txt → 3
            try:
                doc_id = int(filename.replace('speech_', '').replace('.txt', ''))
                filepath = os.path.join(folder_path, filename)
                with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                    documents[doc_id] = f.read()
            except ValueError:
                continue  # skip files that don't match the pattern
    print(f"Loaded {len(documents)} documents.")
    return documents

SPEECHES_FOLDER = r"C:\Users\Ibad Ur Rehman\Desktop\IR_Assignment1\Trump Speechs\Trump Speechs"
documents = load_documents(SPEECHES_FOLDER)

# 
first_id = min(documents.keys())
print(f"\nDocument {first_id} preview (first 300 chars):")
print(documents[first_id][:300])

Loaded 56 documents.

Document 0 preview (first 300 chars):
Remarks Announcing Candidacy for President in New York City
Trump: Wow. Whoa. That is some group of people. Thousands.So nice, thank you very much. That's really nice. Thank you. It's great to be at Trump Tower. It's great to be in a wonderful city, New York. And it's an honor to have everybody here


In [24]:
#making inverted index


def build_inverted_index(documents, stopwords):
   
    #for each document, we preprocess its text and record which terms appear in it.
    
    inverted_index = {}

    for doc_id, text in documents.items():
        tokens = preprocess(text, stopwords)
        unique_terms = set(tokens)  # we only care if term EXISTS in doc (Boolean model)

        for term in unique_terms:
            if term not in inverted_index:
                inverted_index[term] = []
            inverted_index[term].append(doc_id)

    for term in inverted_index:
        inverted_index[term] = sorted(inverted_index[term])

    print(f"Inverted index built with {len(inverted_index)} unique terms.")
    return inverted_index

inverted_index = build_inverted_index(documents, stopwords)

#let's test some termsss
preview_terms = ['trump', 'run', 'america']
for term in preview_terms:
    stemmed = stemmer.stem(term)
    if stemmed in inverted_index:
        print(f"\n'{term}' (stemmed: '{stemmed}') → found in docs: {inverted_index[stemmed]}")

Inverted index built with 4508 unique terms.

'trump' (stemmed: 'trump') → found in docs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 19, 20, 21, 22, 24, 26, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41, 43, 45, 46, 47, 48, 49, 50, 51, 52, 54, 55]

'run' (stemmed: 'run') → found in docs: [0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 27, 28, 30, 32, 33, 34, 35, 36, 37, 39, 40, 41, 44, 45, 46, 47, 50, 51, 52, 53]

'america' (stemmed: 'america') → found in docs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55]


In [25]:
#saving inverted index to disk
#why json???
    #we save as JSON so we don't have to rebuild every time.
    #later we can just load this file directly.

def save_index(index, filepath):
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(index, f, indent=2)
    print(f"Index saved to '{filepath}'")

def load_index(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        index = json.load(f)
    print(f"Index loaded from '{filepath}'")
    return index

save_index(inverted_index, "inverted_index.json")

#testingg
loaded_index = load_index("inverted_index.json")
print(f"Verified: loaded index has {len(loaded_index)} terms.")

Index saved to 'inverted_index.json'
Index loaded from 'inverted_index.json'
Verified: loaded index has 4508 terms.


In [50]:
#now onto positional index


def build_positional_index(documents, stopwords):
   #har document mai har term ki position pata lagaeynge 
    positional_index = {}

    for doc_id, text in documents.items():
        token_positions = preprocess_with_positions(text, stopwords)

        for position, term in token_positions:
            if term is None:
                continue                                        
            if term not in positional_index:
                positional_index[term] = {}

            if doc_id not in positional_index[term]:
                positional_index[term][doc_id] = []

            positional_index[term][doc_id].append(position)

    print(f"Positional index built with {len(positional_index)} unique terms.")
    return positional_index

positional_index = build_positional_index(documents, stopwords)

#testt
preview_terms = ['trump', 'united', 'america']
for term in preview_terms:
    stemmed = stemmer.stem(term)
    if stemmed in positional_index:
        #show positions in first 2 documents only
        preview = dict(list(positional_index[stemmed].items())[:2])
        print(f"\n'{term}' (stemmed: '{stemmed}') → {preview}")

Positional index built with 4508 unique terms.

'trump' (stemmed: 'trump') → {0: [9, 37, 187, 190, 191, 449, 742, 744, 753, 755, 895, 897, 1010, 1013, 1076, 1133, 1136, 1139, 1142, 1143, 1344, 1620, 1621, 1622, 1623, 1624, 1625, 1626, 1776, 1777, 1811, 1910, 2475, 2969, 3804, 3974, 4209, 4709, 4731, 4745, 4842, 5038, 5545, 5661, 5681, 5913, 5984, 5997, 6052, 6209, 6227, 6361, 6499, 6566, 6757, 6850, 6862], 1: [2212]}

'united' (stemmed: 'unit') → {0: [1961, 2092, 2960, 3570, 4173, 6730], 1: [887, 1010, 1047, 1050, 1073, 1163, 1173, 1289, 1401, 1438, 1704, 2171, 2191, 2217, 2270]}

'america' (stemmed: 'america') → {0: [402, 1677, 2927, 2933, 5666, 5669, 5911, 6886], 1: [265, 1076, 1707, 2075, 2247]}


In [51]:
# now save Positional Index to disk
# since JSON requires dict keys to be strings, so doc_ids which are rn integers will be saved as strings.

def save_positional_index(index, filepath):
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(index, f, indent=2)
    print(f"Positional index saved to '{filepath}'")

def load_positional_index(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    #convert string keys back to integers
    index = {}
    for term, doc_dict in raw.items():
        index[term] = {int(doc_id): positions for doc_id, positions in doc_dict.items()}
    print(f"Positional index loaded from '{filepath}'")
    return index

save_positional_index(positional_index, "positional_index.json")

#testt
loaded_positional = load_positional_index("positional_index.json")
print(f"Verified: loaded positional index has {len(loaded_positional)} terms.")

Positional index saved to 'positional_index.json'
Positional index loaded from 'positional_index.json'
Verified: loaded positional index has 4508 terms.


In [28]:
#boolean queries setup


ALL_DOC_IDS = set(documents.keys())

def get_posting(term):
    
    #this function returns the posting list for a term
    stemmed = preprocess_query_term(term)
    if stemmed is None or stemmed not in inverted_index:
        return set()
    return set(inverted_index[stemmed])

def boolean_AND(term1, term2):
    #documents that contain BOTH term1 and term2.
    return get_posting(term1) & get_posting(term2)

def boolean_OR(term1, term2):
    #documents that contain EITHER term1 or term2.
    return get_posting(term1) | get_posting(term2)

def boolean_NOT(term):
    #documents that do NOT contain term.
    return ALL_DOC_IDS - get_posting(term)

print("Boolean operations defined!")

Boolean operations defined!


In [29]:
#query parser

def parse_and_execute(query):
    #what's parsing?
        #Parses a boolean query string and returns matching doc IDs as a sorted list.
    
    query = query.strip()

    #proximity query kelie alag handler hai
    if '/' in query:
        return proximity_query(query)

    #NOT
    if query.upper().startswith('NOT '):
        term = query[4:].strip()
        return sorted(boolean_NOT(term))

    #complex query with parenthesis
    paren_match = re.match(
        r'^(\w+)\s+(AND|OR)\s+\((\w+)\s+(AND|OR)\s+(\w+)(?:\s+(AND|OR)\s+(\w+))?\)$',
        query, re.IGNORECASE
    )
    if paren_match:
        t1 = paren_match.group(1)
        outer_op = paren_match.group(2).upper()
        t2 = paren_match.group(3)
        inner_op = paren_match.group(4).upper()
        t3 = paren_match.group(5)
        t4 = paren_match.group(7)  # optional 4th term

        
        if inner_op == 'AND':
            inner_result = get_posting(t2) & get_posting(t3)
        else:
            inner_result = get_posting(t2) | get_posting(t3)

        
        if t4:
            if inner_op == 'AND':
                inner_result = inner_result & get_posting(t4)
            else:
                inner_result = inner_result | get_posting(t4)

        
        outer_posting = get_posting(t1)
        if outer_op == 'AND':
            result = outer_posting & inner_result
        else:
            result = outer_posting | inner_result

        return sorted(result)

    #chained query
    if ' AND ' in query.upper() or ' OR ' in query.upper():
        parts = re.split(r'\s+(AND|OR)\s+', query, flags=re.IGNORECASE)
        
        terms = parts[0::2]   # every even index is a term
        ops   = parts[1::2]   # every odd index is an operator

        result = get_posting(terms[0])
        for i, op in enumerate(ops):
            next_set = get_posting(terms[i + 1])
            if op.upper() == 'AND':
                result = result & next_set
            else:
                result = result | next_set

        return sorted(result)

    #single term query
    return sorted(get_posting(query))


print("Query parser defined!")

Query parser defined!


In [30]:
#test Queries Against Gold Standard

test_queries = [
    "running",
    "NOT hammer",
    "actions AND wanted",
    "united OR plane",
    "pakistan OR afganistan OR aid",
    "biggest AND ( near OR box )",
    "box AND ( united OR year )",
    "biggest AND ( plane OR wanted OR hour )",
]

print("=" * 60)
for q in test_queries:
    # Clean up spacing around parentheses
    clean_q = q.replace('( ', '(').replace(' )', ')')
    result = parse_and_execute(clean_q)
    print(f"\nQuery : {q}")
    print(f"Result: {set(result)}")
print("=" * 60)


Query : running
Result: {0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 27, 28, 30, 32, 33, 34, 35, 36, 37, 39, 40, 41, 44, 45, 46, 47, 50, 51, 52, 53}

Query : NOT hammer
Result: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 22, 23, 26, 28, 29, 30, 31, 32, 37, 38, 41, 44, 47, 48, 52, 55}

Query : actions AND wanted
Result: {0, 1, 2, 3, 5, 7, 9, 12, 15, 16, 17, 19, 24, 26, 28, 29, 31, 37, 39, 40, 41, 42, 51, 53, 54}

Query : united OR plane
Result: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 54, 55}

Query : pakistan OR afganistan OR aid
Result: {3, 37, 39, 40, 41, 42, 16, 22, 29}

Query : biggest AND ( near OR box )
Result: {4, 6, 43, 44, 45, 46, 47, 18, 50, 51, 53, 54}

Query : box AND ( united OR year )
Result: {4, 9, 44, 45, 46, 47, 18, 50, 54, 23, 25}

Query : biggest AND ( plane OR wanted OR hour )

In [53]:
#dealing with proximity queries

def proximity_query(query):
    #steps:
    #1. Parse the query to extract term1, term2, and k
    #2. Find docs that contain BOTH terms (using inverted index)
    #3. For each such doc, check if any position pair is within k distance
    
    parts = query.strip().split('/')
    k = int(parts[1].strip())
    terms = parts[0].strip().split()

    if len(terms) != 2:
        print("Proximity query must have exactly 2 terms.")
        return []

    term1, term2 = terms[0], terms[1]

    #preprocess both terms the same way as documents
    stemmed1 = preprocess_query_term(term1)
    stemmed2 = preprocess_query_term(term2)

    if stemmed1 is None or stemmed2 is None:
        print("One or both terms are stopwords.")
        return []

    #check both terms exist in positional index
    if stemmed1 not in positional_index or stemmed2 not in positional_index:
        return []

    #find docs that contain BOTH terms
    docs1 = set(positional_index[stemmed1].keys())
    docs2 = set(positional_index[stemmed2].keys())
    common_docs = docs1 & docs2

    result = []

    for doc_id in common_docs:
        positions1 = positional_index[stemmed1][doc_id]
        positions2 = positional_index[stemmed2][doc_id]

        #check if any pair of positions is within k distance
        found = False
        for p1 in positions1:
            for p2 in positions2:
                if abs(p1 - p2) <= k + 1:
                    found = True
                    break
            if found:
                break

        if found:
            result.append(doc_id)

    return sorted(result)

print("Proximity query function defined!")

Proximity query function defined!


In [54]:
#Let's test Proximity Queries Against Gold Standard

proximity_queries = [
    "after years /1",
    "develop solutions /1",
    "keep out /2",
]

# Gold standard from your query list
gold_standard = {
    "after years /1":       {6, 7, 44},
    "develop solutions /1": {5, 32},
    "keep out /2":          {20, 24, 39, 40, 51},
}

for q in proximity_queries:
    result = set(proximity_query(q))
    expected = gold_standard[q]
    match = "MATCH" if result == expected else "MISMATCH"
    print(f"\nQuery    : {q}")
    print(f"Expected : {expected}")
    print(f"Got      : {result}")
    print(f"Status   : {match}")



Query    : after years /1
Expected : {44, 6, 7}
Got      : {3, 6, 7, 39, 10, 11, 44, 47}
Status   : MISMATCH

Query    : develop solutions /1
Expected : {32, 5}
Got      : {32, 5}
Status   : MATCH

Query    : keep out /2
Expected : {51, 20, 39, 40, 24}
Got      : {39, 8, 40, 12, 45, 14, 46, 16, 51, 20, 24, 26, 27}
Status   : MISMATCH


In [39]:
# Debug — check what our stemmer produces for each term
# and whether those stemmed terms exist in the positional index

debug_terms = [
    ("after", "years"),
    ("develop", "solutions"),
    ("keep", "out")
]

for t1, t2 in debug_terms:
    s1 = preprocess_query_term(t1)
    s2 = preprocess_query_term(t2)
    
    in_pos1 = s1 in positional_index if s1 else False
    in_pos2 = s2 in positional_index if s2 else False
    
    print(f"\n'{t1}' → stemmed: '{s1}' | in positional index: {in_pos1}")
    print(f"'{t2}' → stemmed: '{s2}' | in positional index: {in_pos2}")
    
    if in_pos1 and in_pos2:
        common = set(positional_index[s1].keys()) & set(positional_index[s2].keys())
        print(f"  Common docs: {sorted(common)}")


'after' → stemmed: 'after' | in positional index: True
'years' → stemmed: 'year' | in positional index: True
  Common docs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 18, 19, 20, 21, 24, 25, 26, 27, 28, 29, 32, 33, 34, 35, 36, 37, 39, 40, 41, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 54]

'develop' → stemmed: 'develop' | in positional index: True
'solutions' → stemmed: 'solut' | in positional index: True
  Common docs: [3, 5, 31, 32]

'keep' → stemmed: 'keep' | in positional index: True
'out' → stemmed: 'out' | in positional index: True
  Common docs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 16, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30, 32, 33, 34, 35, 36, 37, 39, 40, 41, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 54, 55]


In [40]:
# Debug 2 — check actual positions for each query in expected docs

print("=== after years /1 ===")
s1, s2 = 'after', 'year'
for doc_id in [6, 7, 44]:
    p1 = positional_index[s1].get(doc_id, [])
    p2 = positional_index[s2].get(doc_id, [])
    pairs = [(a, b, abs(a-b)) for a in p1 for b in p2 if abs(a-b) <= 3]
    print(f"  Doc {doc_id}: closest pairs (showing dist<=3): {pairs[:5]}")

print("\n=== develop solutions /1 ===")
s1, s2 = 'develop', 'solut'
for doc_id in [5, 32]:
    p1 = positional_index[s1].get(doc_id, [])
    p2 = positional_index[s2].get(doc_id, [])
    pairs = [(a, b, abs(a-b)) for a in p1 for b in p2 if abs(a-b) <= 3]
    print(f"  Doc {doc_id}: closest pairs (showing dist<=3): {pairs[:5]}")

print("\n=== keep out /2 ===")
s1, s2 = 'keep', 'out'
for doc_id in [20, 24, 39, 40, 51]:
    p1 = positional_index[s1].get(doc_id, [])
    p2 = positional_index[s2].get(doc_id, [])
    pairs = [(a, b, abs(a-b)) for a in p1 for b in p2 if abs(a-b) <= 4]
    print(f"  Doc {doc_id}: closest pairs (showing dist<=4): {pairs[:5]}")

=== after years /1 ===
  Doc 6: closest pairs (showing dist<=3): [(225, 227, 2)]
  Doc 7: closest pairs (showing dist<=3): [(718, 720, 2), (767, 769, 2)]
  Doc 44: closest pairs (showing dist<=3): [(863, 864, 1)]

=== develop solutions /1 ===
  Doc 5: closest pairs (showing dist<=3): [(26, 28, 2)]
  Doc 32: closest pairs (showing dist<=3): [(204, 206, 2)]

=== keep out /2 ===
  Doc 20: closest pairs (showing dist<=4): [(491, 494, 3), (503, 506, 3)]
  Doc 24: closest pairs (showing dist<=4): [(874, 877, 3)]
  Doc 39: closest pairs (showing dist<=4): [(39, 43, 4), (1472, 1474, 2)]
  Doc 40: closest pairs (showing dist<=4): [(191, 195, 4), (1376, 1378, 2)]
  Doc 51: closest pairs (showing dist<=4): [(111, 114, 3), (904, 908, 4), (944, 948, 4)]
